In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
import pandas as pd
sample = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")
test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

In [3]:
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from torch.optim import AdamW

# Config

In [5]:
MODEL_NAME  = "microsoft/deberta-v3-small"
MAX_LEN     = 256
BATCH_SIZE  = 8      
EPOCHS      = 3
LR          = 2e-5
LABEL_MAP   = {'A':0, 'B':1, 'C':2, 'D':3, 'E':4}
DEVICE      = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset

In [6]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        prompt  = row['prompt']
        options = [row['A'], row['B'], row['C'], row['D'], row['E']]
        label   = LABEL_MAP[row['answer']]
        
        encodings = [
            self.tokenizer(prompt, opt, truncation=True,
                           max_length=self.max_len, padding="max_length",
                           return_tensors="pt")
            for opt in options
        ]
        input_ids      = torch.stack([e["input_ids"].squeeze()      for e in encodings])  
        attention_mask = torch.stack([e["attention_mask"].squeeze() for e in encodings])
        
        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "labels":         torch.tensor(label, dtype=torch.long)
        }


In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME,torch_dtype=torch.float32).to(DEVICE)

dataset   = MCQDataset(train, tokenizer, MAX_LEN)
loader    = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(loader),          
    num_training_steps=len(loader) * EPOCHS
)

config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/286M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForMultipleChoice LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias                 

model.safetensors:   0%|          | 0.00/286M [00:00<?, ?B/s]

In [8]:
# from torch.cuda.amp import autocast, GradScaler
# scaler = GradScaler()

# for epoch in range(EPOCHS):
#     model.train()
#     total_loss, correct = 0, 0

#     for batch in loader:
#         input_ids      = batch["input_ids"].to(DEVICE)       # [B, 5, seq]
#         attention_mask = batch["attention_mask"].to(DEVICE)
#         labels         = batch["labels"].to(DEVICE)          # [B]

#         outputs = model(input_ids=input_ids,
#                         attention_mask=attention_mask,
#                         labels=labels)   # ← loss computed INTERNALLY here

#         loss = outputs.loss              # CrossEntropyLoss over 5-class logits
#         loss.backward()                  # ← backpropagation

    #     torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    #     optimizer.step()                 # ← weight update
    #     scheduler.step()
    #     optimizer.zero_grad()

    #     total_loss += loss.item()
    #     preds = outputs.logits.argmax(dim=-1)
    #     correct += (preds == labels).sum().item()

    # acc = correct / len(df)
    # print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Top-1 Acc: {acc:.4f}")

In [9]:
from torch.cuda.amp import autocast, GradScaler
scaler = GradScaler()

for epoch in range(EPOCHS):
    model.train()
    total_loss, correct = 0, 0

    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(DEVICE)       # [B, 5, seq]
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["labels"].to(DEVICE)          # [B]

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels)   # ← loss computed INTERNALLY here

        loss = outputs.loss              # CrossEntropyLoss over 5-class logits
        loss.backward()                  # ← backpropagation
        optimizer.step()                 # ← weight update
        scheduler.step()
        

        total_loss += loss.item()
        preds = outputs.logits.argmax(dim=-1)
        correct += (preds == labels).sum().item()

    acc = correct / len(train)
    print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Top-1 Acc: {acc:.4f}")

/tmp/ipykernel_58/2479329036.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()


Epoch 1 | Loss: 1.4961 | Top-1 Acc: 0.3245
Epoch 2 | Loss: 0.5628 | Top-1 Acc: 0.7905
Epoch 3 | Loss: 0.2352 | Top-1 Acc: 0.9135


In [12]:
# for name, p in model.named_parameters():
#     if p.requires_grad:
#         print(name, p.dtype)

In [10]:
# from torch.cuda.amp import autocast, GradScaler
# scaler = GradScaler()

# for epoch in range(EPOCHS):
#     model.train()
#     total_loss, correct = 0, 0

#     for batch in loader:
#         input_ids = batch["input_ids"].to(DEVICE)
#         attention_mask = batch["attention_mask"].to(DEVICE)
#         labels = batch["labels"].to(DEVICE)

#         optimizer.zero_grad()

#         with autocast():
#             outputs = model(
#                 input_ids=input_ids,
#                 attention_mask=attention_mask,
#                 labels=labels
#             )
#             loss = outputs.loss

    #     scaler.scale(loss).backward()
    #     # scaler.unscale_(optimizer)
    #     # torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    #     scaler.step(optimizer)
    #     scaler.update()
    #     scheduler.step()

    #     total_loss += loss.item()
    #     preds = outputs.logits.argmax(dim=-1)
    #     correct += (preds == labels).sum().item()

    # acc = correct / len(df)
    # print(f"Epoch {epoch+1} | Loss: {total_loss/len(loader):.4f} | Top-1 Acc: {acc:.4f}")

# Testing with one training sample

In [34]:
model.to('cuda')
model.eval()
with torch.no_grad():
    # Example: predict top-3 for first question
    sample   = dataset[4]
    ids      = sample["input_ids"].unsqueeze(0).to(DEVICE)
    attn     = sample["attention_mask"].unsqueeze(0).to(DEVICE)
    logits   = model(input_ids=ids, attention_mask=attn).logits
    probs    = F.softmax(logits, dim=-1)
    top3_p, top3_i = torch.topk(probs, k=3, dim=-1)
    options  = ['A','B','C','D','E']
    print("Top-3 guesses:", [options[i] for i in top3_i[0].tolist()])
    print("Confidences:  ", top3_p[0].tolist())

Top-3 guesses: ['A', 'B', 'C']
Confidences:   [0.9998238682746887, 8.346704998984933e-05, 4.690103742177598e-05]


In [23]:
MAX_LEN  = 256       
BATCH_SZ = 8
TOP_N    = 3
LABEL_MAP_INV = {0: 'A', 1: 'B', 2: 'C', 3: 'D', 4: 'E'}

class MCQTestDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row     = self.df.iloc[idx]
        prompt  = row['prompt']
        options = [row['A'], row['B'], row['C'], row['D'], row['E']]

        encodings = [
            self.tokenizer(prompt, opt,
                           truncation=True,
                           max_length=self.max_len,
                           padding="max_length",
                           return_tensors="pt")
            for opt in options
        ]

        input_ids      = torch.stack([e["input_ids"].squeeze()      for e in encodings])  # [5, seq]
        attention_mask = torch.stack([e["attention_mask"].squeeze() for e in encodings])  # [5, seq]

        return {
            "input_ids":      input_ids,
            "attention_mask": attention_mask,
            "id":             row["id"],
        }

test_dataset = MCQTestDataset(test, tokenizer, MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SZ, shuffle=False)

In [24]:
model.eval()

all_ids         = []
all_top_n_labels = []
all_top_n_probs  = []

with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch["input_ids"].to(DEVICE)      # [B, 5, seq]
        attention_mask = batch["attention_mask"].to(DEVICE) # [B, 5, seq]
        ids            = batch["id"].tolist()               # Python ints

        outputs = model(input_ids=input_ids,
                        attention_mask=attention_mask)      # logits: [B, 5]
        probs   = F.softmax(outputs.logits, dim=-1)         # [B, 5]

        top_probs, top_idx = torch.topk(probs, k=3, dim=-1)  # [B, TOP_N]

        for i, qid in enumerate(ids):
            labels_i = [LABEL_MAP_INV[j.item()] for j in top_idx[i]]
            probs_i  = [round(p.item(), 4) for p in top_probs[i]]

            all_ids.append(qid)
            all_top_n_labels.append(labels_i)
            all_top_n_probs.append(probs_i)

In [31]:
submission = pd.DataFrame({
    "ID":        all_ids,
    "Prediction": [" ".join(labels) for labels in all_top_n_labels],  # e.g. "B A C"
})
submission.to_csv("submission.csv", index=False)
print(submission.head())

   ID Prediction
0   1      A D C
1   2      B D C
2   3      B E D
3   4      E C A
4   5      C D A


In [30]:
# sample1 = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")
# sample1

In [35]:
import os
save_dir = "/kaggle/working/mcq_deberta_model"
os.makedirs(save_dir, exist_ok=True)

model.to("cpu")
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

print("Saved model to:", save_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to: /kaggle/working/mcq_deberta_model
